In [ ]:
### RAG pipeline

In [ ]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [ ]:
def process_all_pdfs(pdf_directory):
    all_documents = []
    pdf_dir = Path(pdf_directory)

    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} pdf files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing files: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'

            all_documents.extend(documents)
            print(f" all documents loaded {len(documents)} pages")

        except Exception as e:
            print(f" error: {e}")

    print(f"\n Total documents loaded: {len(all_documents)}")

    return all_documents


In [ ]:
all_pdf_documents = process_all_pdfs("../data/text_files/pdf")

In [ ]:
all_pdf_documents

In [ ]:
### text splitting

def split_documents(documents, chunk_size=1000, chunk_overlap=200):

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        add_start_index=True,
        separators=["\n\n", "\n", " ", ""]
    )

    split_docs = text_splitter.split_documents(documents)
    print(f"split {len(documents)} documents into {len(split_docs)} chunks")

    if split_docs:
        print(f"Example chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs


In [ ]:
chunks = split_documents(all_pdf_documents)

In [ ]:
print("total chunks:", len(chunks))

for i, chunk in enumerate(chunks[:5]):
    print(f"\nchunls {i+1}")
    print("Length:", len(chunk.page_content))
    print("Page:",chunk.metadata.get("page"))
    print("Content:", chunk.page_content)

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import hashlib
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
class EmbeddingManager:
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):

        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):

        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded succesfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model : {self.model_name}: {e}")
            raise


    def generate_embeddings(self, texts: List[str]) -> np.ndarray:

        if self.model is None:
            raise ValueError("Model not loaded")


        print(f"Generating embedding for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings        


           

In [ ]:
embedding_manager = EmbeddingManager()
embedding_manager

### vector store

In [ ]:
def generate_chunk_id(doc):
    """Create a deterministic ID for this chunk and its location in the source."""
    source = doc.metadata.get("source") or doc.metadata.get("source_file") or "unknown_source"
    # Normalize paths so the same file resolves to the same source key in this environment.
    source = str(Path(str(source)).resolve())
    page = str(doc.metadata.get("page", ""))
    start_index = str(doc.metadata.get("start_index", ""))
    content = doc.page_content

    identity = "\0".join([source, page, start_index, content])
    return hashlib.sha256(identity.encode("utf-8")).hexdigest()


class VectorStore:
    def __init__(
        self,
        collection_name: str = "pdf_documents",
        persist_directory: str = "../data/textfiles/vector_store"
    ):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized: {self.collection_name}")
            print(f"Existing records in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """Insert or update chunks by stable ID; rerunning with the same chunks is safe."""
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match the number of embeddings")
        if not documents:
            print("No documents to add.")
            return

        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for doc, embedding in zip(documents, embeddings):
            ids.append(generate_chunk_id(doc))

            metadata = dict(doc.metadata)
            # Avoid an order-dependent doc_index; keep stable descriptive metadata instead.
            metadata["content_length"] = len(doc.page_content)
            metadatas.append(metadata)
            documents_text.append(doc.page_content)
            embeddings_list.append(np.asarray(embedding, dtype=float).tolist())

        # Guard against an unexpected duplicate ID within the current input batch.
        if len(ids) != len(set(ids)):
            raise ValueError(
                "Duplicate stable IDs found in this batch. Check source/page/start_index metadata."
            )

        try:
            before = self.collection.count()
            self.collection.upsert(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            after = self.collection.count()
            print(f"Upserted {len(documents)} chunks.")
            print(f"Collection count before: {before}; after: {after}")
        except Exception as e:
            print(f"Error upserting documents into vector store: {e}")
            raise

    def reset_collection(self):
        """Delete and recreate this collection. This permanently removes its records."""
        self.client.delete_collection(name=self.collection_name)
        self.collection = self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={"description": "PDF document embeddings for RAG"}
        )
        print(f"Reset collection '{self.collection_name}'. Current count: {self.collection.count()}")


vectorstore = VectorStore()
vectorstore


In [ ]:
chunks

### One-time cleanup of old random-ID records

The old ingestion code generated random UUIDs, so the current persistent collection may already contain duplicates.
After updating the splitter and `VectorStore` code, run the following line **once** to clear the old collection, then rerun the embedding and ingestion cell below. This permanently deletes records in this collection, so do not run it if you need to preserve them.

```python
# vectorstore.reset_collection()  # Uncomment and run once only, if safe to delete existing records.
```

After the reset, run the splitter cell again so chunks include `start_index`, then run the embedding/ingestion cell. Future reruns of the same corpus use `upsert()` and stable IDs.

In [ ]:
texts = [doc.page_content for doc in chunks]

if not texts:
    raise ValueError("No text chunks found. Check the PDF directory and splitting step.")

embeddings = embedding_manager.generate_embeddings(texts)
vectorstore.add_documents(chunks, embeddings)


In [ ]:
print("Total records:", vectorstore.collection.count())

results = vectorstore.collection.get(
    limit=3,
    include=["documents", "metadatas"]
)

for i, text in enumerate(results["documents"]):
    print(f"\nDocument {i + 1}:")
    print(text[:200])
    print("Metadata:", results["metadatas"][i])

### Retrivel

In [ ]:
class RAGRetriver:

    def __init__(
        self,
        vector_store: VectorStore,
        embedding_manager: EmbeddingManager
    ):
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager
    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) ->List[Dict[str, Any]]:

        print(f"Retreving document fro query: '{query}'")
        print(f" Top_k : {top_k}, Score threshold: {score_threshold}")

        query_embedding =self.embedding_manager.generate_embeddings([query])[0]

        try: 
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )

            retrieved_docs = []

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):

                    similarity_score = 1 / (1 + distance)

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            "id": doc_id,
                            "content": document,
                            "metadata": metadata,
                            "similarity_score": similarity_score,
                            "distance": distance,
                            "rank": i+1
                        })

                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")


            else:
                print("No documents found")

            return retrieved_docs

        except Exception as e:
            print(f"Error during retrievel : {e}")

            return []

rag_retriver = RAGRetriver(vectorstore, embedding_manager) 


In [ ]:
rag_retriver

In [ ]:
result = rag_retriver.retrieve("what is deep learning")

In [ ]:
print(result[0])

In [ ]:
results = rag_retriver.retrieve(
    "What is deep learning?",
    top_k=5
)

for result in results:
    print(f"\nRank: {result['rank']}")
    print(f"Score: {result['similarity_score']:.4f}")
    print(f"Page: {result['metadata'].get('page_label')}")
    print(f"Content:\n{result['content'][:300]}")
    print("-" * 60)

In [ ]:
results = rag_retriver.retrieve(
    "What is deep learning?",
    top_k=5
)

for result in results:
    print(
        result["rank"],
        result["id"],
        result["metadata"].get("source_file"),
        result["metadata"].get("page_label"),
        result["content"][:100].replace("\n", " ")
    )